In [1]:
import pandas as pd
from pathlib import Path

RUTA = Path('../data/raw')
print(pd.__version__)

3.0.6


## Preparación: carga de datos

In [2]:
orders      = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
order_prods = pd.read_csv(RUTA / 'order_products.csv', sep=';')
products    = pd.read_csv(RUTA / 'products.csv', sep=';')
aisles      = pd.read_csv(RUTA / 'aisles.csv', sep=';')
departments = pd.read_csv(RUTA / 'departments.csv', sep=';')

In [3]:
for nombre, df in [('orders', orders), ('order_prods', order_prods), ('products', products),
                   ('aisles', aisles), ('departments', departments)]:
    print(nombre, df.shape)

orders (478967, 6)
order_prods (4545007, 4)
products (49694, 4)
aisles (134, 2)
departments (21, 2)


# Lección 2: Limpieza y estandarización de datos crudos

En esta lección se resuelven los problemas encontrados en la exploración: duplicados, tipos de dato incorrectos, nulos, texto sin estandarizar y duplicados implícitos.

Para cada problema hay tres opciones: **eliminar**, **corregir** o **conservar**, y la decisión depende de lo que significa el dato.

Orden de trabajo: duplicados → tipos → nulos → texto → duplicados implícitos → verificación.

## 1. Duplicados exactos
Primero se **detectan** con `duplicated()` (solo marca, no borra) y luego se **eliminan** con `drop_duplicates()`.

In [4]:
duplicados = orders[orders.duplicated(keep=False)].sort_values('order_id')
print(len(duplicados))
duplicados[['order_id', 'order_dow', 'order_hour_of_day']].head(6)

30


,order_id,order_dow,order_hour_of_day
354993,391768,3,2
371905,391768,3,2
119251,408114,3,2
321100,408114,3,2
394347,467134,3,2
250626,467134,3,2


In [5]:
orders = orders.drop_duplicates().reset_index(drop=True)
print(orders.shape, orders['order_id'].is_unique)

(478952, 6) True


**Resultado:** 30 filas marcadas = 15 pedidos duplicados. Todos ocurren el miércoles (`order_dow = 3`) a las 2 a. m.
**Decisión:** eliminar.
**Justificación:** que todos caigan en la misma franja indica un fallo del sistema, no pedidos reales.

## 2. Tipos de dato con `astype`
Se cambian los enteros de 64 bits por tipos más pequeños según el rango de cada columna, para reducir la memoria.

In [6]:
print('Antes:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

orders = orders.astype({
    'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
    'order_dow': 'int8', 'order_hour_of_day': 'int8',
    'days_since_prior_order': 'Int16',
})
order_prods = order_prods.astype({
    'order_id': 'int32', 'product_id': 'int32',
    'add_to_cart_order': 'Int16', 'reordered': 'int8',
})

print('Después:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

Antes: 23.0 MB
Después: 7.2 MB


**Resultado:** la memoria de `orders` bajó de 23,0 MB a 7,2 MB.
Las columnas con nulos (`days_since_prior_order` y `add_to_cart_order`) usan `Int16` (mayúscula) porque es el tipo entero que admite valores nulos.

## 3. Nulos: comprobar la causa antes de decidir
Un nulo puede significar "no aplica", "no se registró" o "error".

In [7]:
nulos = orders['days_since_prior_order'].isna()
primeros = orders['order_number'] == 1
print((nulos == primeros).all())

True


**Decisión:** conservar los nulos de `days_since_prior_order`.
**Justificación:** cada nulo es un primer pedido; no existe pedido anterior. Rellenar con 0 diría que el cliente volvió el mismo día y bajaría la media de espera.

In [8]:
print(products[products['product_name'].isna()][['aisle_id', 'department_id']].value_counts())
products['product_name'] = products['product_name'].fillna('Unknown')
print(products['product_name'].isna().sum())

aisle_id  department_id
100       21               1258
Name: count, dtype: int64
0


**Decisión:** rellenar `product_name` con `'Unknown'`. Los 1.258 productos están en el pasillo 100 y el departamento 21 (`missing`); el producto existe y aparece en pedidos, no se puede borrar.

| Decisión | Columna | Causa |
|---|---|---|
| Conservar el nulo | `days_since_prior_order` | Primer pedido del cliente |
| Rellenar con `'Unknown'` | `product_name` | Producto sin datos de catálogo |
| Conservar el nulo | `add_to_cart_order` | Se investiga en el reto |

## 4. Estandarización de texto con `.str`

In [9]:
nombres = products['product_name']
print('Espacios dobles:', nombres.str.contains('  ', regex=False).sum())
print('Espacio invisible:', nombres.str.contains('\xa0', regex=False).sum())

products['product_name'] = (
    products['product_name']
    .str.replace('\xa0', ' ', regex=False)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    .str.title()
)
print(products['product_name'].str.contains('  ', regex=False).sum(),
      products['product_name'].str.contains('\xa0', regex=False).sum())

Espacios dobles: 403
Espacio invisible: 16
0 0


**Resultado:** 403 nombres tenían espacios dobles y 16 un espacio invisible (`\xa0`). Tras la limpieza quedan 0 y 0.
Se reemplazó el espacio invisible, se unificaron los espacios, se quitaron los de los extremos y se puso mayúscula inicial en cada palabra.

## 5. Duplicados implícitos
Nombres que solo se repiten después de estandarizar el texto.

In [10]:
conocidos = products[products['product_name'] != 'Unknown']
repetidos = conocidos[conocidos['product_name'].duplicated(keep=False)]
print(conocidos['product_name'].duplicated().sum())
print(repetidos['product_id'].isin(order_prods['product_id']).sum(), 'de', len(repetidos))

141
268 de 280


**Decisión:** conservar y documentar.
**Justificación:** tienen `product_id` distintos y 268 de 280 aparecen en pedidos; borrarlos dejaría pedidos apuntando a productos inexistentes. En los análisis se agrupará por `product_id`, no por nombre.

## 6. Limpieza reutilizable con funciones

In [11]:
def limpiar_orders(df):
    df = df.drop_duplicates().reset_index(drop=True)
    df = df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                    'order_dow': 'int8', 'order_hour_of_day': 'int8',
                    'days_since_prior_order': 'Int16'})
    assert df['order_id'].is_unique
    return df

def limpiar_products(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

print(limpiar_orders(orders).shape)

(478952, 6)


In [12]:
assert orders.shape == (478952, 6)
assert orders['days_since_prior_order'].isna().sum() == 28817   # 2 nulos estaban entre los duplicados
assert products['product_name'].isna().sum() == 0
assert str(order_prods['add_to_cart_order'].dtype) == 'Int16'

## Resumen
1. Se revisaron y eliminaron 15 pedidos duplicados con `duplicated(keep=False)` y `drop_duplicates`.
2. Se corrigieron los tipos con `astype`, usando `Int16` en las columnas con nulos.
3. Se analizó la causa de los nulos: se conservaron en `days_since_prior_order` y se rellenó `product_name` con `'Unknown'`.
4. Se estandarizó el texto con `.str.replace`, `.str.strip` y `.str.title`.
5. Se detectaron 141 duplicados implícitos y se conservaron porque se usan en pedidos.
6. Se empaquetó la limpieza en funciones y se verificó con `assert`.

# Laboratorio guiado 2: Pipeline de limpieza con informe de cambios

Se construye un pipeline: una secuencia de pasos de limpieza encadenados que parte del archivo crudo, se ejecuta con una sola instrucción y deja un informe de lo que cambió en cada paso.

## Paso 1: Recargar los datos crudos

In [13]:
orders_crudo   = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')
op_crudo       = pd.read_csv(RUTA / 'order_products.csv', sep=';')
print(orders_crudo.shape, products_crudo.shape, op_crudo.shape)

(478967, 6) (49694, 4) (4545007, 4)


## Paso 2: Función que anota el estado de la tabla
Después de cada paso se toma una "foto" de la tabla: filas, nulos y memoria.

In [14]:
registro = []

def anotar(df, tabla, paso):
    registro.append({
        'tabla': tabla,
        'paso': paso,
        'filas': len(df),
        'nulos': int(df.isna().sum().sum()),
        'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 1),
    })
    return df

## Paso 3: Un paso de limpieza por función
Cada función hace una sola cosa: recibe un DataFrame y devuelve otro.

In [15]:
def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)

def tipos_orders(df):
    return df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                      'order_dow': 'int8', 'order_hour_of_day': 'int8',
                      'days_since_prior_order': 'Int16'})

def limpiar_nombres(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

def tipos_order_prods(df):
    return df.astype({'order_id': 'int32', 'product_id': 'int32',
                      'add_to_cart_order': 'Int16', 'reordered': 'int8'})

## Paso 4: Encadenar los pasos con `pipe`

In [16]:
orders_limpio = (orders_crudo
    .pipe(anotar, 'orders', '0 crudo')
    .pipe(quitar_duplicados)
    .pipe(anotar, 'orders', '1 sin duplicados')
    .pipe(tipos_orders)
    .pipe(anotar, 'orders', '2 tipos'))

products_limpio = (products_crudo
    .pipe(anotar, 'products', '0 crudo')
    .pipe(limpiar_nombres)
    .pipe(anotar, 'products', '1 nombres'))

op_limpio = (op_crudo
    .pipe(anotar, 'order_prods', '0 crudo')
    .pipe(tipos_order_prods)
    .pipe(anotar, 'order_prods', '1 tipos'))

## Paso 5: Informe de cambios

In [17]:
informe = pd.DataFrame(registro)
informe

,tabla,paso,filas,nulos,memoria_mb
0,orders,0 crudo,478967,28819,23.0
1,orders,1 sin duplicados,478952,28817,23.0
2,orders,2 tipos,478952,28817,7.2
3,products,0 crudo,49694,1258,5.1
4,products,1 nombres,49694,0,5.1
5,order_prods,0 crudo,4545007,836,145.4
6,order_prods,1 tipos,4545007,836,54.5


**Interpretación del informe:**
- **Quitar duplicados** eliminó 15 filas de `orders` y 2 nulos: dos de los pedidos repetidos eran primeros pedidos.
- **Corregir tipos** no cambió filas ni nulos, pero redujo la memoria de `orders` de 23,0 a 7,2 MB y la de `order_prods` de 145,4 a 54,5 MB.
- **Limpiar nombres** dejó `products` con 0 nulos (los 1.258 se rellenaron con `'Unknown'`).
- Los **836 nulos** de `order_prods` siguen ahí a propósito: se decidió conservarlos.

## Paso 6: Validar el resultado
El pipeline termina con una comprobación automática de las reglas que deben cumplir los datos limpios.

In [18]:
def validar(o, p, op):
    assert o['order_id'].is_unique, 'order_id repetido'
    assert o['order_hour_of_day'].between(0, 23).all(), 'hora fuera de rango'
    assert p['product_name'].notna().all(), 'productos sin nombre'
    assert not p['product_name'].str.contains('\xa0', regex=False).any(), 'espacios invisibles'
    assert op['reordered'].isin([0, 1]).all(), 'reordered con valores raros'
    return 'Validación superada'

print(validar(orders_limpio, products_limpio, op_limpio))
print(orders_limpio.equals(orders), products_limpio.equals(products))

Validación superada
True True


## Conclusiones clave
- Un pipeline es una cadena de funciones pequeñas que reciben y devuelven un DataFrame, unidas con `pipe`.
- Intercalar una función de registro (`anotar`) produce un informe de cambios sin esfuerzo extra.
- El informe responde con cifras a la pregunta "¿qué se le hizo a los datos?".
- Validar con `assert` al final convierte las reglas de calidad en una prueba que se ejecuta cada vez.

# Reto 2

## Ejercicio 1
Comprueba si `order_prods` tiene filas duplicadas exactas, si se repite algún par `order_id` + `product_id` y si `reordered` solo contiene 0 y 1.

In [19]:
print('Filas duplicadas exactas:', order_prods.duplicated().sum())
print('Pares order_id + product_id repetidos:', order_prods.duplicated(subset=['order_id', 'product_id']).sum())
print('Valores de reordered:', order_prods['reordered'].unique())

Filas duplicadas exactas: 0
Pares order_id + product_id repetidos: 0
Valores de reordered: [0 1]


**Resultado:** 0 filas duplicadas exactas y 0 pares `order_id` + `product_id` repetidos; `reordered` solo contiene 0 y 1.
**Conclusión:** `order_prods` no necesita limpieza de duplicados: cada producto aparece una sola vez por pedido, y `reordered` es una variable binaria válida.

## Ejercicio 2
Comprueba si `aisles` y `departments` tienen nombres duplicados o con espacios al inicio o al final. ¿Necesitan limpieza de texto?

In [20]:
print('Pasillos duplicados:', aisles['aisle'].duplicated().sum())
print('Pasillos con espacios en los extremos:', (aisles['aisle'] != aisles['aisle'].str.strip()).sum())

print('Departamentos duplicados:', departments['department'].duplicated().sum())
print('Departamentos con espacios en los extremos:', (departments['department'] != departments['department'].str.strip()).sum())

Pasillos duplicados: 0
Pasillos con espacios en los extremos: 0
Departamentos duplicados: 0
Departamentos con espacios en los extremos: 0


**Resultado:** `aisles` tiene 0 nombres duplicados y 0 con espacios en los extremos; `departments` también 0 y 0.
**Conclusión:** no necesitan limpieza de texto. Son tablas de referencia pequeñas (134 y 21 filas) que ya vienen limpias.

## Ejercicio 3
Investiga los 836 nulos de `add_to_cart_order`: ¿en cuántos pedidos ocurren y cuántos productos tiene, como mínimo, cada uno? ¿Qué los explica?

In [21]:
filas_nulas = order_prods[order_prods['add_to_cart_order'].isna()]
print('Filas con nulo:', len(filas_nulas))

pedidos_con_nulo = filas_nulas['order_id'].unique()
print('Pedidos con nulo:', len(pedidos_con_nulo))

detalle = order_prods[order_prods['order_id'].isin(pedidos_con_nulo)]
productos_por_pedido = detalle.groupby('order_id').size()
print('Mínimo de productos en esos pedidos:', productos_por_pedido.min())
print('Posición máxima registrada en todo order_prods:', order_prods['add_to_cart_order'].max())

Filas con nulo: 836
Pedidos con nulo: 70
Mínimo de productos en esos pedidos: 65
Posición máxima registrada en todo order_prods: 64


**Resultado:** los 836 nulos ocurren en 70 pedidos, y cada uno tiene como mínimo 65 productos.
**Explicación:** la posición más alta registrada en toda la tabla es 64. Todos los pedidos con nulos superan los 64 productos, así que el sistema solo guardaba la posición en el carrito hasta el producto 64; desde el 65 en adelante quedó vacía.
**Decisión:** conservar el nulo. El producto sí fue comprado; lo que falta es solo su posición, y no hay forma confiable de reconstruirla.

## Ejercicio 4
Carga de nuevo `products.csv` en una variable `products_crudo` y cuenta cuántos nombres (no nulos) cambiaron con la estandarización.

In [22]:
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')

no_nulos = products_crudo['product_name'].notna()
cambiaron = (products_crudo.loc[no_nulos, 'product_name'] != products.loc[no_nulos, 'product_name']).sum()
print('Nombres que cambiaron:', cambiaron)

Nombres que cambiaron: 6321


**Resultado:** 6.321 nombres cambiaron con la estandarización (espacios dobles, espacios invisibles, espacios en los extremos o mayúsculas).
Se excluyeron los nulos porque los 1.258 productos sin nombre ahora dicen `'Unknown'` y se contarían como cambio sin que hubiera un nombre que corregir.

## Ejercicio 5
Crea una Series con el nombre del día de cada pedido y conviértela a una categoría ordenada de domingo a sábado. Compara la memoria del texto con la de la categoría.|

In [23]:
dias = {0: 'Domingo', 1: 'Lunes', 2: 'Martes', 3: 'Miércoles',
        4: 'Jueves', 5: 'Viernes', 6: 'Sábado'}

dia_texto = orders['order_dow'].map(dias)

orden = ['Domingo', 'Lunes', 'Martes', 'Miércoles', 'Jueves', 'Viernes', 'Sábado']
dia_categoria = pd.Series(pd.Categorical(dia_texto, categories=orden, ordered=True))

print('Texto:', round(dia_texto.memory_usage(deep=True) / 1e6, 1), 'MB')
print('Categoría:', round(dia_categoria.memory_usage(deep=True) / 1e6, 1), 'MB')
dia_categoria.head()

Texto: 28.6 MB
Categoría: 0.5 MB


0     Sábado
1    Viernes
2    Viernes
3    Domingo
4     Jueves
dtype: category
Categories (7, str): ['Domingo' < 'Lunes' < 'Martes' < 'Miércoles' < 'Jueves' < 'Viernes' < 'Sábado']

**Resultado:** como texto ocupa 28.6 MB; como categoría ocupa 0,5 MB, unas 57.2 veces menos.
**Explicación:** el texto guarda la palabra completa en cada una de las 478.952 filas. La categoría guarda los 7 nombres una sola vez y en cada fila solo un número pequeño que apunta a uno de ellos. Además, al ser ordenada, al agrupar u ordenar los días aparecen de domingo a sábado y no en orden alfabético.

## Ejercicio 6
¿Cuántas filas de `order_prods` corresponden a productos `'Unknown'` y en cuántos pedidos aparecen? ¿Los eliminarías? Justifica.

In [24]:
ids_unknown = products.loc[products['product_name'] == 'Unknown', 'product_id']

filas_unknown = order_prods[order_prods['product_id'].isin(ids_unknown)]
print('Filas:', len(filas_unknown))
print('Pedidos:', filas_unknown['order_id'].nunique())

Filas: 9547
Pedidos: 8254


**Resultado:** 9.547 filas de `order_prods` corresponden a productos `'Unknown'`, repartidas en 8.254 pedidos.
**¿Los eliminaría?:** no.
**Justificación:** son compras reales; al producto solo le falta el nombre en el catálogo. Eliminarlos dejaría 8.254 pedidos incompletos y reduciría el total de artículos, la tasa de recompra y el tamaño de los carritos. Es mejor conservarlos con la etiqueta `'Unknown'` y excluirlos solo en los análisis que dependan del nombre del producto.

## Ejercicio 7
En una copia de `departments`, reemplaza `'missing'` por un nulo real (`pd.NA`). Explica en una frase por qué no conviene hacerlo en la tabla original antes de agrupar.

In [25]:
departments_copia = departments.copy()
departments_copia['department'] = departments_copia['department'].replace('missing', pd.NA)

print(departments_copia.tail(3))
print('Original con missing:', (departments['department'] == 'missing').sum())

    department_id department
18             19     snacks
19             20       deli
20             21        NaN
Original con missing: 1


**Explicación:** no conviene hacerlo en la tabla original porque `groupby` descarta los nulos por defecto (`dropna=True`): los 1.258 productos del departamento 21 desaparecerían en silencio de cualquier resumen por departamento, mientras que con la etiqueta `'missing'` siguen apareciendo como un grupo visible.

# Lección 3: Filtrado, ordenamiento y agrupación de datos

Se convierten las tablas limpias en respuestas: cuándo compran los clientes y con qué frecuencia vuelven. **Filtrar** elige las filas que importan, **ordenar** las jerarquiza y **agrupar** las resume.

Antes de escribir código, cada pregunta se traduce a la frase "métrica por grupo". Por ejemplo, "¿A qué hora se compra más?" se convierte en "número de pedidos por hora".

## 1. Filtrado con máscaras booleanas

In [26]:
es_primer_pedido = orders['order_number'] == 1
print(es_primer_pedido.head(3))
primeros = orders[es_primer_pedido]
print(len(primeros))

0    False
1    False
2    False
Name: order_number, dtype: bool
28817


In [27]:
es_fin_de_semana = orders['order_dow'].isin([0, 6])
es_manana = orders['order_hour_of_day'].between(6, 11)
print(len(orders[es_fin_de_semana & es_manana]))

45599


**Resultado:** hay 28.817 primeros pedidos y 45.599 pedidos hechos el fin de semana entre las 6 y las 11 a. m.
Las máscaras se combinan con `&` (y), `|` (o) y `~` (no); no se usan `and` ni `or`.

## 2. Filtrado con `query`

In [29]:
nocturnos_30 = orders.query('order_hour_of_day >= 22 and days_since_prior_order == 30')
print(len(nocturnos_30))

1449


**Resultado:** 1.449 pedidos se hicieron desde las 10 p. m. por clientes que tardaron 30 días en volver.
`query` permite escribir la condición como texto y es más legible cuando hay varias condiciones.

## 3. Ordenamiento con `sort_values` y `nlargest`

In [30]:
print(orders.nlargest(3, 'order_number')[['user_id', 'order_number']])
historial = orders.sort_values(['user_id', 'order_number'])
print(historial[['user_id', 'order_number']].head(3))

      user_id  order_number
2482     8220           100
4552   164906           100
6402   172294           100
        user_id  order_number
237012        2             1
317136        2             4
337239        4             2


**Resultado:** el número de pedido más alto es 100. Al ordenar por cliente y número de pedido se obtiene el historial de compras de cada cliente en orden.
`nlargest` / `nsmallest` sacan los extremos sin ordenar toda la tabla; `sort_values` ordena por una o varias columnas.

## 4. Agrupación con `groupby`
Pregunta: ¿a qué hora se compra más? → **número de pedidos por hora**.

In [31]:
pedidos_por_hora = orders.groupby('order_hour_of_day')['order_id'].count()
print(pedidos_por_hora.idxmax(), pedidos_por_hora.max())

10 40578


In [32]:
resumen_dia = (orders
    .groupby('order_dow')
    .agg(pedidos=('order_id', 'count'),
         clientes=('user_id', 'nunique'),
         espera_media=('days_since_prior_order', 'mean'))
    .round(1))
resumen_dia

,pedidos,clientes,espera_media
order_dow,,,
0,84090,57661,11.8
1,82185,56479,11.3
2,65833,48587,11.2
3,60897,45331,10.7
4,59810,44281,10.5
5,63488,46127,10.5
6,62649,45604,11.4


**Resultado:** la hora pico es las 10 a. m., con 40.578 pedidos. El domingo (0) es el día con más pedidos (84.090) y clientes (57.661), seguido del lunes (82.185). La espera media entre pedidos está entre 10,5 y 11,8 días.
Otras funciones de agregación: `'size'` (filas, incluidos nulos), `'sum'`, `'median'`, `'min'`, `'max'`, `'first'` y `'last'`.

## 5. Tabla dinámica con `pivot_table`

In [33]:
matriz = orders.pivot_table(index='order_hour_of_day', columns='order_dow',
                            values='order_id', aggfunc='count')
print(matriz.shape)
matriz.loc[[8, 10, 14, 20]]

(24, 7)


order_dow,0,1,2,3,4,5,6
order_hour_of_day,,,,,,,
8,3908,4751,3485,3125,3136,3373,3246
10,6978,7881,5489,5026,4938,5347,4919
14,7533,6545,5344,4774,4795,5265,5375
20,2544,2332,2061,1917,2037,1886,1847


**Resultado:** la matriz tiene 24 filas (horas) y 7 columnas (días). La combinación con más pedidos es el lunes a las 10 a. m., con 7.881 pedidos.
`pivot_table` cruza dos variables en una sola tabla, como una tabla dinámica de Excel.

## 6. Valores por fila con `transform`

In [34]:
orders['pedidos_cliente'] = orders.groupby('user_id')['order_id'].transform('count')
frecuentes = orders[orders['pedidos_cliente'] >= 5]
print(len(frecuentes), frecuentes['user_id'].nunique())
orders = orders.drop(columns='pedidos_cliente')

231946 30527


**Resultado:** 30.527 clientes tienen 5 pedidos o más y suman 231.946 pedidos.
`transform` calcula un valor por grupo y lo devuelve en cada fila; `agg` devuelve una fila por grupo.

## Resumen de la explicación
1. Filtrado con máscaras booleanas y los operadores `&`, `|` y `~`.
2. Filtros con `isin`, `between` y `query`.
3. Ordenamiento con `sort_values`, `nlargest` y `nsmallest`.
4. Agrupación con `groupby` y agregación con nombre usando `agg`.
5. Tablas dinámicas con `pivot_table`.
6. Cálculo de valores de grupo por fila con `transform`.

# Laboratorio guiado 3: Perfil de hábitos de compra

El área de mercadeo quiere diseñar campañas según qué tan seguido compra cada cliente y necesita saber a qué hora compra cada tipo de cliente.

Plan:
1. Contar los pedidos de cada cliente.
2. Clasificar a los clientes en tres segmentos.
3. Clasificar los pedidos en franjas horarias.
4. Cruzar segmentos y franjas.

## Paso 1: Pedidos por cliente
Pregunta: número de pedidos por cliente.

In [35]:
pedidos_por_cliente = orders.groupby('user_id')['order_id'].count()
print(len(pedidos_por_cliente))
pedidos_por_cliente.describe()

157437


count    157437.000000
mean          3.042182
std           2.746842
min           1.000000
25%           1.000000
50%           2.000000
75%           4.000000
max          28.000000
Name: order_id, dtype: float64

## Paso 2: Segmentar clientes con `pd.cut`
Segmentos: **ocasional** (1 pedido), **regular** (2 a 4) y **frecuente** (5 o más).

In [36]:
import numpy as np

segmento_cliente = pd.cut(pedidos_por_cliente,
                          bins=[0, 1, 4, np.inf],
                          labels=['ocasional', 'regular', 'frecuente'])
print(segmento_cliente.value_counts().sort_index())
print((segmento_cliente.value_counts(normalize=True).sort_index() * 100).round(1))

order_id
ocasional    55357
regular      71553
frecuente    30527
Name: count, dtype: int64
order_id
ocasional    35.2
regular      45.4
frecuente    19.4
Name: proportion, dtype: float64


## Paso 3: Llevar el segmento a cada pedido con `map`
El segmento está calculado por cliente, pero se necesita en cada pedido.

In [37]:
orders['segmento'] = orders['user_id'].map(segmento_cliente)
orders[['user_id', 'order_number', 'segmento']].head()

,user_id,order_number,segmento
0,183418,11,regular
1,163593,5,regular
2,39980,4,ocasional
3,82516,56,frecuente
4,196724,2,ocasional


## Paso 4: Resumen por segmento

In [38]:
resumen_segmento = (orders
    .groupby('segmento', observed=True)
    .agg(clientes=('user_id', 'nunique'),
         pedidos=('order_id', 'count'),
         espera_mediana=('days_since_prior_order', 'median')))
resumen_segmento

,clientes,pedidos,espera_mediana
segmento,,,
ocasional,55357,55357,16.0
regular,71553,191649,11.0
frecuente,30527,231946,6.0


**Interpretación:** el 19,4 % de los clientes (los frecuentes) hace casi la mitad de los pedidos (231.946 de 478.952) y vuelve cada 6 días (mediana). Los regulares vuelven cada 11 días y los ocasionales tienen una espera de 16 días: aunque solo tienen un pedido en estos datos, ese pedido no es necesariamente el primero que hicieron en la tienda.

## Paso 5: Franjas horarias y tabla cruzada
Se clasifica cada pedido según su hora y se cruza con el segmento del cliente.

In [39]:
orders['franja'] = pd.cut(orders['order_hour_of_day'],
                          bins=[-1, 5, 11, 17, 23],
                          labels=['madrugada', 'mañana', 'tarde', 'noche'])

perfil = (pd.crosstab(orders['segmento'], orders['franja'], normalize='index')
            .mul(100).round(1))
perfil

franja,madrugada,mañana,tarde,noche
segmento,,,,
ocasional,2.1,30.0,49.4,18.5
regular,1.9,31.6,48.4,18.1
frecuente,1.7,35.2,45.9,17.2


**Interpretación:** todos los segmentos compran sobre todo en la tarde (entre 45,9 % y 49,4 %), pero los clientes frecuentes compran más en la mañana (35,2 %) que los ocasionales (30,0 %). Para mercadeo, una campaña de fidelización debería enviarse en la mañana.

## Paso 6: Combinaciones de día y franja más activas

In [40]:
(orders.groupby(['order_dow', 'franja'], observed=True)
       .size()
       .sort_values(ascending=False)
       .head(3))

order_dow  franja
0          tarde     42340
1          tarde     37432
2          tarde     30751
dtype: int64

**Resultado:** las combinaciones más activas son domingo en la tarde (42.340 pedidos), lunes en la tarde (37.432) y martes en la tarde (30.751).

## Paso 7: Dejar `orders` como estaba
Las columnas auxiliares ya cumplieron su función y se eliminan para no arrastrarlas a la Lección 4.

In [41]:
orders = orders.drop(columns=['segmento', 'franja'])
print(orders.shape)

(478952, 6)


## Conclusiones clave
- Una pregunta de negocio compleja se resuelve en pasos: agrupar, categorizar, devolver al detalle y cruzar.
- `pd.cut` convierte números en categorías con límites explícitos.
- `map` lleva un resultado calculado por grupo a cada fila del detalle.
- `crosstab` con `normalize` permite comparar grupos de distinto tamaño.

# Reto 3
Para cada punto se escribe primero la pregunta como "métrica por grupo" o como condición, y después el código.

## Ejercicio 1
¿Cuáles son las tres horas con más pedidos los domingos y cuántos pedidos tiene cada una?

**Traducción:** condición `order_dow == 0` → número de pedidos por hora → las 3 más altas.

In [42]:
domingos = orders[orders['order_dow'] == 0]
pedidos_hora_domingo = domingos.groupby('order_hour_of_day')['order_id'].count()
pedidos_hora_domingo.nlargest(3)

order_hour_of_day
15    7585
14    7533
13    7483
Name: order_id, dtype: int64

**Resultado:** los domingos, las horas con más pedidos son las 3 p. m. (7.585), las 2 p. m. (7.533) y la 1 p. m. (7.483).
**Interpretación:** el domingo el pico de compras está en la tarde temprana, a diferencia del total de la semana, cuyo pico es a las 10 a. m.

## Ejercicio 2
¿A qué hora es mayor la espera media entre pedidos (`days_since_prior_order`) y a qué hora es menor?

**Traducción:** espera media por hora → la hora del valor máximo y la del mínimo.

In [43]:
espera_por_hora = orders.groupby('order_hour_of_day')['days_since_prior_order'].mean()
print('Mayor espera:', espera_por_hora.idxmax(), 'h ->', round(espera_por_hora.max(), 2), 'días')
print('Menor espera:', espera_por_hora.idxmin(), 'h ->', round(espera_por_hora.min(), 2), 'días')

Mayor espera: 0 h -> 12.52 días
Menor espera: 22 h -> 10.62 días


**Resultado:** la mayor espera media está a las 0 h (medianoche), con 12,52 días; la menor, a las 22 h (10 p. m.), con 10,62 días.
**Interpretación:** quienes compran a las 10 p. m. vuelven unos 2 días antes que quienes compran a medianoche.

## Ejercicio 3
¿Cuántos clientes tienen 10 pedidos o más?

**Traducción:** número de pedidos por cliente → condición `>= 10` → contar clientes.

In [44]:
pedidos_cliente = orders.groupby('user_id')['order_id'].count()
print((pedidos_cliente >= 10).sum())

6327


**Resultado:** 6.327 clientes tienen 10 pedidos o más (alrededor del 4 % de los 157.437 clientes).
**Interpretación:** es un grupo pequeño de clientes muy fieles dentro del segmento frecuente.

## Ejercicio 4
¿Cuántos pedidos se hicieron exactamente 7 días después del anterior y qué porcentaje del total representan?

**Traducción:** condición `days_since_prior_order == 7` → contar → dividir entre el total de pedidos.

In [45]:
semanales = (orders['days_since_prior_order'] == 7).sum()
porcentaje = semanales / len(orders) * 100
print(semanales, round(porcentaje, 2), '%')

44577 9.31 %


**Resultado:** 44.577 pedidos se hicieron exactamente 7 días después del anterior, el 9,31 % del total.
**Interpretación:** casi 1 de cada 10 pedidos sigue un ciclo semanal; es un patrón de compra de mercado cada semana.

## Ejercicio 5
Usando `order_prods`: ¿cuántos pedidos tienen 20 productos o más? ¿Cuál es el pedido más grande y cuántos productos tiene?

**Traducción:** número de productos por pedido → condición `>= 20` → contar; y el valor máximo.

In [46]:
productos_por_pedido = order_prods.groupby('order_id').size()
print('Pedidos con 20 productos o más:', (productos_por_pedido >= 20).sum())
print('Pedido más grande:', productos_por_pedido.idxmax(), 'con', productos_por_pedido.max(), 'productos')

Pedidos con 20 productos o más: 49420
Pedido más grande: 61355 con 127 productos


**Resultado:** 49.420 pedidos tienen 20 productos o más. El pedido más grande es el 61355, con 127 productos.
**Interpretación:** este pedido supera el límite de 64 posiciones que vimos en la Lección 2, por lo que parte de sus productos tiene nulo en `add_to_cart_order`.

## Ejercicio 6
Usando `order_prods`: calcula la tasa de recompra (media de `reordered`) para las posiciones 1 a 5 del carrito. ¿Qué patrón ves?

**Traducción:** condición `add_to_cart_order` entre 1 y 5 → media de `reordered` por posición.

In [47]:
primeras_posiciones = order_prods[order_prods['add_to_cart_order'].between(1, 5)]
tasa_por_posicion = primeras_posiciones.groupby('add_to_cart_order')['reordered'].mean().round(3)
tasa_por_posicion

add_to_cart_order
1    0.679
2    0.677
3    0.659
4    0.638
5    0.617
Name: reordered, dtype: float64

**Resultado:** la tasa de recompra baja con cada posición: 0,679 en la posición 1, 0,677 en la 2, 0,659 en la 3, 0,638 en la 4 y 0,617 en la 5.
**Patrón:** los primeros productos que el cliente agrega al carrito son los que más recompra; los clientes empiezan por sus productos habituales y después agregan productos nuevos.

## Ejercicio 7
Compara la mediana de `days_since_prior_order` de los pedidos de fin de semana (domingo y sábado) con la del resto de la semana.

**Traducción:** mediana de la espera por grupo (fin de semana sí / no).

In [48]:
es_fin_de_semana = orders['order_dow'].isin([0, 6])
orders.groupby(es_fin_de_semana)['days_since_prior_order'].median()

order_dow
False    7.0
True     8.0
Name: days_since_prior_order, dtype: Float64

**Resultado:** la mediana de la espera es de 8 días en los pedidos de fin de semana y de 7 días en los del resto de la semana.
**Interpretación:** quienes compran el fin de semana tardan, típicamente, un día más en volver que quienes compran entre semana.